# Local Retrieval Mini Project – Youssef

## Objective
Build a local retrieval system from scratch.

### Pipeline
1. Load dataset
2. Preprocess text
3. TF-IDF vectorization
4. Cosine similarity
5. Top-k retrieval
6. Confidence threshold

## Dataset

Use `course_catalog.csv` as the local knowledge base (72 lessons). `support_questions.csv` is reserved for later evaluation.

In [1]:
from pathlib import Path

DATA_DIR = Path("data")

for file in DATA_DIR.glob("*.csv"):
    print(file.name)

support_questions.csv
course_catalog.csv


In [2]:
import csv

DATA_PATH = DATA_DIR / "course_catalog.csv"

with open(DATA_PATH, encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    courses = list(reader)

print("Total courses:", len(courses))
print("\nColumns:", list(courses[0].keys()))
print("\nFirst course:", courses[0])

Total courses: 72

Columns: ['lesson_id', 'title', 'module', 'description']

First course: {'lesson_id': 'llm_001', 'title': 'Next-Token Prediction Basics', 'module': 'foundations', 'description': 'Explains LLMs as systems that repeatedly predict the next token. Covers probability distributions, generated continuations, why fluent text can still be unsupported, and how application context changes the next-token prediction loop.'}


## Text Preparation

Lowercase and tokenize course text. Remove common stopwords **before** creating the TF-IDF index.

In [3]:
import re

STOPWORDS = {
    "a", "an", "and", "are", "as", "be",
    "can", "do", "does", "for", "from",
    "how", "i", "in", "is", "it", "of",
    "or", "the", "to", "what", "when",
    "where", "why", "with"
}


def tokenize(text):
    tokens = re.findall(r"\b\w+\b", text.lower())
    return [token for token in tokens if token not in STOPWORDS]


print(tokenize("Understanding LLMs and Token Prediction!"))

['understanding', 'llms', 'token', 'prediction']


In [4]:
for course in courses:
    course["search_text"] = " ".join([
        course["title"],
        course["module"],
        course["description"]
    ])

    course["tokens"] = tokenize(course["search_text"])


print("Course:", courses[0]["title"])
print("Tokens:", courses[0]["tokens"])
print("Total tokens:", len(courses[0]["tokens"]))

Course: Next-Token Prediction Basics
Tokens: ['next', 'token', 'prediction', 'basics', 'foundations', 'explains', 'llms', 'systems', 'that', 'repeatedly', 'predict', 'next', 'token', 'covers', 'probability', 'distributions', 'generated', 'continuations', 'fluent', 'text', 'still', 'unsupported', 'application', 'context', 'changes', 'next', 'token', 'prediction', 'loop']
Total tokens: 29


## IDF and TF-IDF

IDF downweights common terms across documents; TF-IDF combines corpus rarity with frequency in a course.

In [5]:
import math
from collections import Counter

# Number of documents
N = len(courses)

# Count in how many documents each word appears
document_frequency = Counter()

for course in courses:
    unique_tokens = set(course["tokens"])
    document_frequency.update(unique_tokens)

# Calculate IDF
idf = {}

for word, frequency in document_frequency.items():
    idf[word] = math.log((N + 1) / (frequency + 1)) + 1

# Inspect selected words
for word in ["the", "token", "embedding", "prediction"]:
    print(word, round(idf.get(word, 0), 3))

the 0
token 3.093
embedding 3.093
prediction 4.597


## create TF-IDF Vectors 



In [6]:
def compute_tfidf(tokens, idf):
    counts = Counter(tokens)
    total = len(tokens)

    if total == 0:
        return {}

    vector = {}

    for word, count in counts.items():
        tf = count / total
        vector[word] = tf * idf.get(word, 0)

    return vector


# Build TF-IDF vectors for all 72 courses
for course in courses:
    course["vector"] = compute_tfidf(course["tokens"], idf)


# Inspect first course
vector = courses[0]["vector"]

for word in ["token", "prediction", "the"]:
    print(word, round(vector.get(word, 0), 4))

token 0.32
prediction 0.3171
the 0


## Similarity and Ranking

Use cosine similarity to compare a query with each course, then return the top-k results.

In [7]:
def cosine_similarity(vec1, vec2):
    # Dot product
    dot_product = sum(
        value * vec2.get(word, 0)
        for word, value in vec1.items()
    )

    # Vector lengths
    norm1 = math.sqrt(sum(v**2 for v in vec1.values()))
    norm2 = math.sqrt(sum(v**2 for v in vec2.values()))

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return dot_product / (norm1 * norm2)


# User query
query = "How does token prediction work?"

# Convert query into TF-IDF using the existing IDF weights
query_tokens = tokenize(query)
query_vector = compute_tfidf(query_tokens, idf)

# Compare with first course
score = cosine_similarity(
    query_vector,
    courses[0]["vector"]
)

print("Query:", query)
print("Course:", courses[0]["title"])
print("Similarity:", round(score, 4))

Query: How does token prediction work?
Course: Next-Token Prediction Basics
Similarity: 0.5056


In [8]:
def search_courses(query, courses, k=3):
    query_tokens = tokenize(query)
    query_vector = compute_tfidf(query_tokens, idf)

    results = []

    for course in courses:
        score = cosine_similarity(
            query_vector,
            course["vector"]
        )

        results.append({
            "title": course["title"],
            "module": course["module"],
            "score": score
        })

    # Sort by highest similarity
    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results[:k]


# Test our search engine
query = "How does token prediction work?"

results = search_courses(query, courses)

for rank, result in enumerate(results, start=1):
    print(
        f"{rank}. {result['title']} "
        f"| Module: {result['module']} "
        f"| Score: {result['score']:.4f}"
    )

1. Next-Token Prediction Basics | Module: foundations | Score: 0.5056
2. Support Router Token Estimate | Module: practice | Score: 0.2129
3. Token-Based Cost Estimation | Module: tokenization | Score: 0.1573


In [9]:
def search_with_threshold(query, courses, k=3, threshold=0.20):

    results = search_courses(query, courses, k=k)

    relevant_results = [
        result
        for result in results
        if result["score"] >= threshold
    ]

    if not relevant_results:
        print("No sufficiently relevant courses found.")
        return []

    return relevant_results


# Test 1: Relevant query
query = "How does token prediction work?"

results = search_with_threshold(query, courses)

for result in results:
    print(result["title"], round(result["score"], 4))

Next-Token Prediction Basics 0.5056
Support Router Token Estimate 0.2129


In [10]:
query = "How can I configure a Kubernetes cluster?"

results = search_with_threshold(query, courses)

print("Number of results:", len(results))

No sufficiently relevant courses found.
Number of results: 0


## Quick Evaluation

Test relevant and unrelated queries. A similarity threshold is a heuristic, **not** a calibrated confidence probability.

In [11]:
test_queries = [
    "How does token prediction work?",
    "What are embeddings?",
    "How can I configure Kubernetes?"
]

for query in test_queries:
    print(f"\nQuery: {query}")

    results = search_with_threshold(
        query,
        courses,
        k=3,
        threshold=0.20
    )

    for result in results:
        print(
            f"  {result['title']}: "
            f"{result['score']:.4f}"
        )


Query: How does token prediction work?
  Next-Token Prediction Basics: 0.5056
  Support Router Token Estimate: 0.2129

Query: What are embeddings?
  Neural Embeddings Overview: 0.4151

Query: How can I configure Kubernetes?
No sufficiently relevant courses found.


## Key Takeaways

- Local retrieval works without an LLM, an API, or a vector database.
- This TF-IDF search is **lexical**, not semantic: word forms such as `embedding` and `embeddings` may not match.
- A threshold of `0.20` is only an initial heuristic. Irrelevant questions may still produce false positives.
- **Next:** evaluate retrieval against labeled questions and tune preprocessing and thresholds.